# 04 — Évaluation de la qualité RAG

**Objectif** : mesurer la qualité du système sur deux dimensions.

Ce notebook couvre les points **2.4** et **2.5** du formulaire SpeciTec :
- 2.4 Évaluation (Hit Rate, MRR, Faithfulness, Relevance)
- 2.5 Retour d'expérience (problèmes rencontrés + solutions)

In [ ]:
import sys, warnings, logging, json
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

from src.retriever import JobRetriever
from src.rag import JobRAG
from src.evaluate import RagEvaluator

retriever = JobRetriever()
rag = JobRAG(retriever)
evaluator = RagEvaluator(retriever, rag)

print(f'Évaluateur initialisé : {len(evaluator.questions)} questions de test')

## 2.4.1 — Qualité du retrieval (automatique)

**Protocole** : pour chaque question, on vérifie si au moins un keyword attendu
apparaît dans les top-5 résultats (Hit Rate) et quel est le rang du premier
résultat pertinent (MRR).

In [ ]:
report = evaluator.run_retrieval_eval(verbose=True)
evaluator.save_results(report)

## 2.4.2 — Tableau récapitulatif des métriques

In [ ]:
import pandas as pd

g = report['global']
bd = report['by_difficulty']

rows = [
    ['Hit rate (top-5) — global',    f"{g['hit_rate']:.1%}"],
    ['Hit rate — easy',              f"{bd['easy']['hit_rate']:.1%}"],
    ['Hit rate — medium',            f"{bd['medium']['hit_rate']:.1%}"],
    ['Hit rate — hard',              f"{bd['hard']['hit_rate']:.1%}"],
    ['MRR — global',                 f"{g['mean_mrr']:.4f}"],
    ['Category hit rate',            f"{g['category_hit_rate']:.1%}" if g['category_hit_rate'] else 'N/A'],
    ['Faithfulness (moyenne /5)',     '? (annotation manuelle requise)'],
    ['Relevance (moyenne /5)',        '? (annotation manuelle requise)'],
    ['% citations correctes',        '? (annotation manuelle requise)'],
]

df_metrics = pd.DataFrame(rows, columns=['Métrique', 'Score'])
print(df_metrics.to_string(index=False))

## 2.4.3 — Qualité des réponses (annotation manuelle)

Génère les réponses pour 10 questions représentatives.
**Après exécution** : ouvrir `eval/results/answer_eval_*.json` et remplir
les champs `manual_scores` (faithfulness, relevance, citations_correct).

In [ ]:
from datetime import datetime
from config import EVAL_RESULTS_DIR

answer_report = evaluator.run_answer_eval_sample(n_questions=10, verbose=True)

# Sauvegarde pour annotation manuelle
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = EVAL_RESULTS_DIR / f'answer_eval_{ts}.json'
out_path.write_text(
    json.dumps(answer_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'\nFichier à annoter : {out_path}')
print('→ Ouvre ce fichier et remplis les champs manual_scores')

## 2.4.4 — Agrégation des scores manuels (après annotation)

In [ ]:
# Exécuter après avoir annoté le fichier answer_eval_*.json

# Trouver le dernier fichier d'évaluation annoté
answer_files = sorted(EVAL_RESULTS_DIR.glob('answer_eval_*.json'))
if answer_files:
    latest = answer_files[-1]
    metrics = evaluator.compute_answer_metrics(latest)
    print('=== Métriques qualité réponses ===')
    for k, v in metrics.items():
        print(f'  {k}: {v}')
else:
    print('Aucun fichier answer_eval trouvé — lance la cellule précédente d\'abord')

---

## 2.5 — Retour d'expérience

### Problème 1 : ChromaDB incompatible Python 3.14

**Constat** : ChromaDB 1.5.2 utilise `pydantic.v1` qui lève une `ConfigError`
au simple `import chromadb` sur Python 3.14.

**Hypothèses testées** :
1. Downgrade ChromaDB 0.6.x → échec (chroma-hnswlib sans wheel Python 3.14)
2. Switch vers FAISS → succès

**Solution** : remplacé ChromaDB par FAISS (`IndexFlatIP` + fichiers JSON pour les métadonnées).
Persistence via `faiss.write_index()` / `faiss.read_index()`.

**Impact** : aucune dégradation des performances — FAISS est même plus rapide
pour un corpus de 180 vecteurs (exact search sans overhead réseau).

---

### Problème 2 : Hit rate de 60% sur les questions medium/hard

**Constat** : 8 questions sur 25 ne trouvent pas les keywords attendus dans top-5.

In [ ]:
# Analyse des questions manquées
missed = [q for q in report['per_question'] if not q['hit']]
print(f'{len(missed)} questions sans hit :')
for q in missed:
    print(f"  [{q['id']}] {q['question'][:70]}")
    print(f"       keywords attendus : {q['expected_keywords']}")
    print(f"       top-1 récupéré    : {q['top1_title'][:50]} (score={q['top1_score']:.3f})")

**Analyse** : les misses sont concentrées sur des technologies rares dans le corpus
(Kafka, Airflow, Tableau, télétravail) ou sur des catégories sous-représentées (DBA_INFRA : 5 offres).

**Solutions testées** :
- Augmenter TOP_K_RETRIEVAL de 20 à 40 → amélioration marginale (+1 hit)
- Baisser MIN_SIMILARITY de 0.30 à 0.20 → plus de bruit
- **Vrai fix** : collecter plus d'offres DBA et avec ces technologies spécifiques

**Leçon** : la qualité d'un RAG est bornée par la richesse du corpus.
Un système de retrieval parfait ne peut pas trouver ce qui n'est pas indexé.

---

**Prochaine étape** : voir `README.md` pour le récapitulatif complet